# Embedding & Vectorization (Pipeline 003)

## Overview

Scheduled notebook that embeds unembedded failed job runs into Databricks Vector Search.

### Pipeline Order
`001` (one-time setup) → `002` (ingestion) → **`003`** (embedding) → `004` (query)

## What This Notebook Does

- **Pulls Unembedded Failures:** Queries `rag_jobs.jobs_run_log` joined with `error_registry` to find runs not yet vectorized

- **Builds Rich Documents:** For each failure, creates a comprehensive text document including:
  - Error type
  - Root cause from error registry
  - Recommended actions
  - Full traceback

- **Generates Embeddings:** Uses `databricks-gte-large-en` (Databricks Foundation Model API)
  - No external API keys required
  - Batch size: 32 requests (respects Foundation Model API rate limits)

- **Stores Vectors:** Upserts embeddings into `rag_jobs.job_embeddings` Delta table
  - Stores 1024-dimensional vectors
  - No external vector database needed
  - Metadata included for retrieval context

- **Marks Complete:** Sets `embedded_at` timestamp on processed rows using `run_id` as unique key

## Execution

- **One-time setup:** Run "Create job_embeddings table" once before scheduling
- **Scheduled:** Run after notebook `002` or on the same cadence (e.g., hourly)
- **Retrieval:** Notebook `004` loads embeddings from Delta and uses cosine similarity for search


In [ ]:
# COMMAND ----------

# DBTITLE 1,Create job_embeddings table (one-time, idempotent)
# DBTITLE 1,Create job_embeddings table (one-time, idempotent)
# Stores embedding vectors alongside metadata for cosine-similarity retrieval in notebook 004.
# Re-running is safe -- CREATE TABLE IF NOT EXISTS is a no-op when the table already exists.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {EMBEDDINGS_TABLE} (
        doc_id     STRING       COMMENT 'str(run_id) -- primary key, unique per run',
        run_id     LONG         COMMENT 'Databricks job run_id',
        job_id     STRING,
        job_name   STRING,
        run_date   STRING,
        error_type STRING,
        severity   STRING,
        email      STRING,
        actions    STRING,
        cluster_id STRING,
        content    STRING       COMMENT 'Full document text that was embedded',
        embedding  ARRAY<FLOAT> COMMENT '1024-dim vector from databricks-gte-large-en',
        created_at TIMESTAMP    COMMENT 'When this record was embedded'
    )
    USING DELTA
    COMMENT 'Job failure embeddings -- source for RAG retrieval in notebook 004'
""")
print(f"\u2705 Table {EMBEDDINGS_TABLE} is ready.")

In [ ]:
# COMMAND ----------

# DBTITLE 1,Config & setup
# DBTITLE 1,Config & setup
# No pip install needed - databricks-sdk and mlflow are pre-installed in DBR 14.3+
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# --- Widget ---
# run_date : optional date filter - blank processes ALL unembedded failures
dbutils.widgets.text("run_date", "", "Run date (dd/MM/yyyy, blank = all unembedded)")

EMBED_MODEL       = "databricks-gte-large-en"  # Databricks Foundation Model API - no API key needed
BATCH_SIZE        = 32    # stay within Foundation Model API per-request limits
EMBEDDINGS_TABLE  = "rag_jobs.job_embeddings"  # Delta table storing vectors in hive_metastore

print(f"Embed model      : {EMBED_MODEL}")
print(f"Embeddings table : {EMBEDDINGS_TABLE}")

In [ ]:
# COMMAND ----------

# DBTITLE 1,Pull unembedded failed jobs
# DBTITLE 1,Pull unembedded failed jobs
from datetime import datetime

input_date = dbutils.widgets.get("run_date").strip()

if input_date:
    filter_date = datetime.strptime(input_date, "%d/%m/%Y").strftime("%Y-%m-%d")
    date_filter = f"AND j.run_date = '{filter_date}'"
    print(f"Date filter   : {filter_date}")
else:
    date_filter = ""  # process ALL unembedded failures - safe to run even if the scheduler missed a day
    print("Date filter   : none - processing all unembedded failed runs")

failed_jobs_df = spark.sql(f"""
    SELECT j.*, e.root_cause, e.actions, e.email, e.severity
    FROM rag_jobs.jobs_run_log j
    LEFT JOIN rag_jobs.error_registry e USING (error_type)
    WHERE j.status    = 'FAILED'
    AND   j.embedded_at IS NULL
    {date_filter}
""")

display(failed_jobs_df)
print(f"Unembedded failed runs: {failed_jobs_df.count()}")

In [ ]:
# COMMAND ----------

# DBTITLE 1,Build document records
# DBTITLE 1,Build document records
from datetime import date

failed_jobs = failed_jobs_df.toPandas().to_dict(orient="records")

def build_record(job: dict) -> dict:
    """Build a flat dict ready for VS upsert from a joined jobs_run_log + error_registry row."""
    run_date_str = job["run_date"].strftime("%d/%m/%Y") if isinstance(job["run_date"], date) else str(job["run_date"])

    content = f"""JOB FAILURE RECORD
==================
Job ID: {job['job_id']}
Job Name: {job['job_name']}
Run Date: {run_date_str}
Cluster: {job['cluster_id']}
Triggered By: {job['triggered_by']}
Duration: {job['duration_secs']} seconds

ERROR TYPE: {job['error_type']}
ERROR MESSAGE: {job['error_message']}

ROOT CAUSE:
{job['root_cause'] or 'Not documented'}

RECOMMENDED ACTIONS:
{job['actions'] or 'No actions registered for this error type'}

NOTIFY: {job['email'] or 'No contact registered'}
SEVERITY: {job['severity'] or 'UNKNOWN'}

FULL TRACEBACK:
{job['traceback']}"""

    return {
        "doc_id":     str(job["run_id"]),          # run_id is unique per run; job_id is NOT (same job can fail many times)
        "run_id":     int(job["run_id"]),
        "job_id":     job["job_id"],
        "job_name":   job["job_name"],
        "run_date":   run_date_str,
        "error_type": job["error_type"] or "UNKNOWN",
        "severity":   job["severity"]   or "UNKNOWN",
        "email":      job["email"]      or "",
        "actions":    (job["actions"]   or "")[:500],
        "cluster_id": job["cluster_id"],
        "content":    content,
    }

records = [build_record(job) for job in failed_jobs]

if not records:
    print("No unembedded failed jobs - nothing to embed.")
    dbutils.notebook.exit("no_records")
else:
    print(f"\u2705 Built {len(records)} document records.")
    print(records[0]["content"][:300])

# COMMAND ----------

# DBTITLE 1,Embed, upsert to Vector Search, stamp embedded_at
# DBTITLE 1,Embed, upsert to job_embeddings, stamp embedded_at
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType, FloatType, TimestampType

def embed_texts(texts: list[str]) -> list[list[float]]:
    """Call databricks-gte-large-en via the SDK serving endpoint (no API key needed)."""
    response = w.serving_endpoints.query(name=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]

# --- Step 1: Embed in batches ---
for i in range(0, len(records), BATCH_SIZE):
    batch = records[i : i + BATCH_SIZE]
    embeddings = embed_texts([r["content"] for r in batch])
    for rec, emb in zip(batch, embeddings):
        rec["embedding"] = [float(v) for v in emb]
        rec["created_at"] = datetime.now()

print(f"\u2705 Embedded {len(records)} documents using {EMBED_MODEL}")

# --- Step 2: MERGE into job_embeddings Delta table ---
schema = StructType([
    StructField("doc_id",     StringType(),            True),
    StructField("run_id",     LongType(),              True),
    StructField("job_id",     StringType(),            True),
    StructField("job_name",   StringType(),            True),
    StructField("run_date",   StringType(),            True),
    StructField("error_type", StringType(),            True),
    StructField("severity",   StringType(),            True),
    StructField("email",      StringType(),            True),
    StructField("actions",    StringType(),            True),
    StructField("cluster_id", StringType(),            True),
    StructField("content",    StringType(),            True),
    StructField("embedding",  ArrayType(FloatType()),  True),
    StructField("created_at", TimestampType(),         True),
])
new_df = spark.createDataFrame(records, schema=schema)
new_df.createOrReplaceTempView("embedding_updates")
spark.sql(f"""
    MERGE INTO {EMBEDDINGS_TABLE} AS target
    USING embedding_updates AS source
    ON target.doc_id = source.doc_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
print(f"\u2705 Upserted {len(records)} vectors into {EMBEDDINGS_TABLE}")

# --- Step 3: Stamp embedded_at using run_id (not job_id - same job can fail multiple times) ---
run_ids_str = ", ".join(str(r["run_id"]) for r in records)
spark.sql(f"""
    UPDATE rag_jobs.jobs_run_log
    SET embedded_at = current_timestamp()
    WHERE run_id IN ({run_ids_str})
""")

print(f"\u2705 Stamped embedded_at for {len(records)} runs in jobs_run_log")


In [ ]:
# COMMAND ----------

# DBTITLE 1,Sanity check - cosine similarity search
# DBTITLE 1,Sanity check - cosine similarity search
# Embeds the test query and ranks all records in job_embeddings by cosine similarity.
# This is the same retrieval pattern used in notebook 004.
import numpy as np

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

test_query = "why did the customer job fail with a null column"

query_emb = np.array(embed_texts([test_query])[0])
emb_pd = spark.table(EMBEDDINGS_TABLE).toPandas()
emb_pd["score"] = emb_pd["embedding"].apply(lambda e: cosine_similarity(query_emb, np.array(e)))
top = emb_pd.nlargest(3, "score")[["score", "job_name", "error_type", "severity", "actions"]]

print(f"Query: '{test_query}'\n")
for _, row in top.iterrows():
    print(f"Score  : {row['score']:.3f}")
    print(f"Job    : {row['job_name']} | Error: {row['error_type']} | Severity: {row['severity']}")
    print(f"Actions: {str(row['actions'])[:120]}...")
    print("---")
if emb_pd.empty:
    print("Table is empty -- check that the embed cell ran successfully.")